# 02 — Exploration de l’agressivité chimique

Objectif : explorer les deux feuilles du classeur, calculer Larson et préparer la modélisation.

## 1. Configuration

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
DATA_RAW=PROJECT_ROOT/"data"/"raw"; DATA_PROCESSED=PROJECT_ROOT/"data"/"processed"
FIGURES=PROJECT_ROOT/"reports"/"figures"; RESULTS=PROJECT_ROOT/"reports"/"results"; MODELS=PROJECT_ROOT/"models"
for p in [DATA_PROCESSED,FIGURES,RESULTS,MODELS]: p.mkdir(parents=True,exist_ok=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RANDOM_STATE=42
pd.set_option("display.max_columns",100)

In [ ]:
from src.data_cleaning import load_water_quality,basic_cleaning,data_quality_report,save_processed
from src.indices_chimiques import add_larson_features

## 2. Chargement des feuilles

In [ ]:
p=DATA_RAW/"water_quality.xlsx"
mhl=basic_cleaning(load_water_quality(p,"MHLATHUZE")); luv=basic_cleaning(load_water_quality(p,"LUVUVU"))
print("MHLATHUZE",mhl.shape,"LUVUVU",luv.shape)
display(mhl.head()); display(luv.head())

## 3. Contrôle qualité

In [ ]:
print(list(mhl.columns)); print(list(luv.columns)); display(data_quality_report(mhl)); display(data_quality_report(luv))

## 4. Colonnes communes

In [ ]:
common=sorted(set(mhl.columns)&set(luv.columns)); chem=[c for c in ["pH","EC","TDS","Na","K","Ca","Mg","Cl","HCO3","SO4","NO3"] if c in common]
print(common); print("Variables retenues :",chem)

## 5. Indice de Larson

$IC=([Cl^-]+2[SO_4^{2-}])/[HCO_3^-]$. Le module convertit les concentrations de mg/L vers meq/L.

In [ ]:
mhl_i=add_larson_features(mhl); luv_i=add_larson_features(luv)
display(mhl_i[["Cl","SO4","HCO3","Larson_index","Larson_class","Larson_corrosive"]].head())

## 6. Distribution des classes

In [ ]:
for name,d in [("MHLATHUZE",mhl_i),("LUVUVU",luv_i)]:
    print(name); display(d["Larson_class"].value_counts(dropna=False).to_frame("effectif")); d["Larson_index"].replace([np.inf,-np.inf],np.nan).dropna().plot.hist(bins=30,figsize=(6,3),title=f"Larson — {name}"); plt.axvline(1,ls="--"); plt.show()

## 7. Corrélation avec Larson

In [ ]:
for name,d in [("MHLATHUZE",mhl_i),("LUVUVU",luv_i)]:
    cols=[c for c in chem+["Larson_index"] if c in d.columns]; print(name); display(d[cols].corr(numeric_only=True)[["Larson_index"]].sort_values("Larson_index",ascending=False))

## 8. Langelier

La température n’est pas présente. Le notebook ne calcule donc pas de LSI avec une température inventée. La fonction est prête dans `src/indices_chimiques.py`.

## 9. Jeu commun

In [ ]:
cols=chem+["Larson_index","Larson_corrosive"]
a=mhl_i[cols].copy(); a["source"]="MHLATHUZE"
b=luv_i[cols].copy(); b["source"]="LUVUVU"
aggr=pd.concat([a,b],ignore_index=True).replace([np.inf,-np.inf],np.nan).dropna(subset=["Larson_corrosive"]).reset_index(drop=True)
display(aggr.head()); print(aggr.shape)

## 10. Export

In [ ]:
save_processed(aggr,DATA_PROCESSED/"aggressiveness_clean.csv")

## 11. Synthèse

Discuter les classes Larson, les différences entre feuilles, l’absence de température pour Langelier et la non-utilisation de `Y` faute de documentation.